# WildChat Pipeline Walkthrough

Run each pipeline stage from this notebook. Stage logic lives in Python scripts so it can also run outside Jupyter. This first version implements Bronze ingestion; raw responses are saved under `data/raw/`, which is excluded from Git. Do not display or commit conversation text.


## 1. Setup and choose the page

The Hugging Face Rows API allows up to 100 rows per request. `OFFSET` selects the first row index. Re-running the fetch creates another timestamped Bronze file.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

BRONZE_SCRIPT_RELATIVE = Path("ingestion/pipelines/bronze/fetch_wildchat.py")
REPO_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / BRONZE_SCRIPT_RELATIVE).is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find the repository root above the notebook working directory.")

OFFSET = 0
LENGTH = 100  # The API allows 1–100 rows per request.
RAW_DIR = REPO_ROOT / "data/raw/bronze/wildchat-1m/train"
BRONZE_SCRIPT = REPO_ROOT / BRONZE_SCRIPT_RELATIVE

print(f"Python: {sys.version.split()[0]}")
print(f"Repository: {REPO_ROOT}")
print(f"Requesting rows {OFFSET}–{OFFSET + LENGTH - 1}")


## 2. Run Bronze ingestion

This runs the same script used outside Jupyter. It saves the raw API response locally and prints only status information.


In [ ]:
existing_files = set(RAW_DIR.glob("rows_offset_*.json"))
result = subprocess.run(
    [
        sys.executable,
        str(BRONZE_SCRIPT),
        "--offset", str(OFFSET),
        "--length", str(LENGTH),
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Bronze ingestion failed. See the error above.")

print(result.stdout.strip())
new_files = set(RAW_DIR.glob("rows_offset_*.json")) - existing_files
if len(new_files) != 1:
    raise RuntimeError("Could not identify the newly written Bronze response.")
bronze_file = new_files.pop()
print(f"Bronze file: {bronze_file.relative_to(REPO_ROOT)}")


## 3. Inspect the response safely

Review counts and one record’s structure without printing conversation text. A record contains a conversation made up of user and assistant messages; moderation results align with those messages.


In [ ]:
payload = json.loads(bronze_file.read_text(encoding="utf-8"))
rows = payload.get("rows", [])
columns = [feature.get("name") for feature in payload.get("features", [])]
row_indexes = [item.get("row_idx") for item in rows if isinstance(item.get("row_idx"), int)]
print(f"Rows received: {len(rows)} / {payload.get('num_rows_total', 'total not provided')} reported")
print(f"Columns: {columns}")
if row_indexes:
    print(f"Row-index range: {min(row_indexes)}–{max(row_indexes)}")

record = rows[0]["row"] if rows else {}
print({
    "model": record.get("model"),
    "language": record.get("language"),
    "turns": record.get("turn"),
    "toxic": record.get("toxic"),
    "redacted": record.get("redacted"),
})
messages = record.get("conversation", [])
openai_results = record.get("openai_moderation", [])
detoxify_results = record.get("detoxify_moderation", [])
print(f"Message count: {len(messages)}")
for i, message in enumerate(messages):
    openai_item = openai_results[i] if i < len(openai_results) else {}
    detoxify_item = detoxify_results[i] if i < len(detoxify_results) else {}
    print(
        f"{i}: role={message.get('role')}, "
        f"content_length={len(message.get('content') or '')}, "
        f"OpenAI fields={list(openai_item)}, "
        f"Detoxify fields={list(detoxify_item)}",
    )


## Next stages

Bronze is the only implemented stage so far. Silver will validate and minimize records; Gold will produce aggregate usage reports. Conversation categorization will follow after the taxonomy is reviewed.
